# InstaNovo Baseline on Nine Species Dataset

This notebook downloads a subset of the **Nine Species Benchmark** and runs the official pre-trained **InstaNovo** model on it. This establishes a baseline that you can use to fairly evaluate your own DFM model.

It uses the official source code located at `~/DiffusionResearchProject/InstaNovo`.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
from datasets import load_dataset

# Ensure InstaNovo is in the python path
INSTANOVO_DIR = Path("~/DiffusionResearchProject/InstaNovo").expanduser()
if str(INSTANOVO_DIR) not in sys.path:
    sys.path.insert(0, str(INSTANOVO_DIR))

In [ ]:
# 1. Download & Subset the Nine Species Dataset
SUBSET_SIZE = 1000  # Adjust this to run on the full dataset (set to 0 for full)
SPLIT = "test"

OUTPUT_DIR = Path("../artifacts/instanovo_baseline")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

parquet_path = OUTPUT_DIR / f"ninespecies_{SPLIT}_subset_{SUBSET_SIZE}.parquet"
predictions_path = OUTPUT_DIR / f"instanovo_predictions_{SUBSET_SIZE}.csv"

if not parquet_path.exists():
    print(f"Downloading ms_ninespecies_benchmark ({SPLIT} split)...")
    ds = load_dataset("InstaDeepAI/ms_ninespecies_benchmark", split=SPLIT)
    
    if SUBSET_SIZE > 0 and SUBSET_SIZE < len(ds):
        print(f"Taking subset of {SUBSET_SIZE} spectra...")
        ds = ds.select(range(SUBSET_SIZE))
        
    ds.to_parquet(parquet_path)
    print(f"Saved to {parquet_path}")
else:
    print(f"Subset already exists at {parquet_path}")

In [ ]:
# 2. Run InstaNovo Inference via CLI
# We use --no-refinement to only run the Transformer model (ignoring the diffusion InstaNovo+ step)
MODEL_ID = "instanovo-v1.2.0"

!PYTHONPATH={INSTANOVO_DIR} python3 -m instanovo.cli predict \
    --data-path {parquet_path} \
    --instanovo-model {MODEL_ID} \
    --output-path {predictions_path} \
    --denovo \
    --no-refinement

In [ ]:
# 3. View the Results
if predictions_path.exists():
    df = pd.read_csv(predictions_path)
    display(df.head(10))
    print(f"\nSaved {len(df)} predictions to {predictions_path}")
else:
    print("Predictions file not found.")